# Week 4 — Memory, Tools, and ReAct with LangChain

This is a beginner-friendly notebook for class. We will learn memory, create tools, and build a simple ReAct-style agent using LangChain concepts.

By the end, you should be able to:
- explain buffer, summary, and window memory
- create simple tools
- understand how ReAct works
- combine these ideas into a small chatbot-like app

## Quick Week 3 Recap

In Week 3, we learned:
- prompts
- chains
- LLM calls
- basic LangChain building blocks

Today we add three new ideas:
1. Memory
2. Tools
3. ReAct

In [1]:
# HINT: Import os and load_dotenv/find_dotenv
# HINT: Load environment variables and print whether the API key is loaded

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override= True)
api_key = os.getenv('OPENAI_API_KEY')

print(bool(api_key))


True


In [10]:
# HINT: Let's check if the langchain memory modules are available
# (Your instructor will run a quick check here to make sure dependencies are set up)

import langchain_community.memory

## Part 1: Memory Basics


### In-Class Exercise 1

Before coding, answer this with your partner:
- If a chatbot must remember your name for the whole session, which memory should it use?
  - Answer: Use buffer or persistent session memory (buffer memory keeps full history for the session).
- If a chat becomes very long, which memory is safer for token limits?
  - Answer: Summary memory (stores a short summary to save tokens).
- If only the last 2 messages matter, which memory is best?
  - Answer: Window memory (ConversationBufferWindowMemory with k=2).

In [23]:
from langchain_core.tools import tool

@tool
def addition(a: float, b: float) -> str:
    """Add two numbers"""
    return str(a+b)

@tool
def subtraction(a: float, b: float) -> str:
    """Subtract second number from first"""
    return str(a - b)

@tool
def multiplication(a: float, b: float) -> str:
    """Multiply 2 numbers"""
    return str(a*b)

@tool
def get_weather(location: str) -> str:
        "Get the weather from Newyork, London, Tokyo only"
        data = {
                "New York": "Sunny, 75F",
                "London": "Rainy, 55F",
                "Tokyo": "Cloudy, 65F"
        }
        return data.get(location.title(), "Weather not found")

@tool
def word_counter(text: str) -> str:
     """Count the number of words in a sentence"""
     words = text.strip().split()
     return str(len(words))

tools = [
     addition,
     subtraction,
     multiplication,
     get_weather,
     word_counter
]

Available tools:
- multiplication : Multiply two numbers.
- hello_message : Return a simple greeting message.
- c_to_f : Convert Celsius to Fahrenheit.
- summarize_paragraph : Summarize a paragraph in a very simple way.


In [13]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0.0)

# Define the Memory Component
memory = InMemorySaver()

# Create an agent with memory:
agent = create_agent(
    model = llm,
    tools = tools,
    checkpointer=memory
)

agent.invoke(
    {
        "messages":[
            {"role": "user", "content": "Hi, my name is Sarvesh"}
        ]
    },
    {
        "configurable": {"thread_id": "student_1"}
    }
)

# Second Message
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is my name?"}
        ]
    },
    {
        "configurable": {"thread_id": "student_1"}
    }
)

print(response["messages"][-1].content)

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Your name is Sarvesh.


In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0.0)

# Define the Memory Component
memory = InMemorySaver()

# Create an agent with memory:
agent = create_agent(
    model = llm,
    tools = tools,
    checkpointer=memory,
    system_prompt = "You are helpful"
)


thread_id = "student_chatbot"
print("Welcome to my Chatbot, please ask your question")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot conversation is over")
        break
    
    response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_input}
        ]
    },
    {
        "configurable": {"thread_id": "student_chatbot"}
    }
)
    
print("Bot:", response['messages'][-1].content)

Welcome to my Chatbot, please ask your question
Chatbot conversation is over
Bot: Your dream salary is 1,000,000.


1. agent
2. add tools
3. Memory
4. break the memory wrt threads
5. Add Streaming
6. Use a real Weather api


### In-Class Exercise 2

Change `ConversationBufferWindowMemory(k=2)` to `ConversationBufferWindowMemory(k=4)` and run again.

Questions to discuss:
- What changed in the stored history?
- Which earlier messages are now visible?
- Why would a real chatbot choose a smaller or larger window?

## LangChain Memory Pattern

In LangChain, a common memory pattern is:
1. create a prompt
2. create a chain
3. store message history
4. inject history into the prompt
5. keep the conversation going

In [21]:
# Use native LangChain memory primitives instead of a custom class
from langchain_community.chat_message_histories import ChatMessageHistory

# Use native ChatMessageHistory instances per session (no custom class)
histories = {}

def get_history(session_id: str) -> ChatMessageHistory:
    return histories.setdefault(session_id, ChatMessageHistory())

# Example usage (students can run):
hist = get_history('student1')
hist.add_user_message('Hi, my name is Ravi.')
hist.add_ai_message('Nice to meet you Ravi.')
hist.add_user_message('I like Python programming.')
print('Stored messages:', [m.content for m in hist.messages])

# Provide default alias for backwards compatibility in demos
mem = get_history('student_demo')

Stored messages: ['Hi, my name is Ravi.', 'Nice to meet you Ravi.', 'I like Python programming.']


## Part 2: Tools

Tools let the model do something outside the chat response.
Examples:
- calculate
- look up data
- format text
- convert units

### In-Class Exercise 4

For each task below, say if a tool is needed:
1. What is 12 x 8?
2. Write a short greeting.
3. Convert 25C to Fahrenheit.
4. Summarize this paragraph.

In [25]:
from langchain_core.tools import tool

@tool
def multiplication(a: float, b: float) -> str:
    """Multiply two numbers."""
    return str(a * b)

@tool
def hello_message(name: str = "Hello") -> str:
    """Return a simple greeting."""
    return "Hello" if not name else f"Hello, {name}!"

@tool
def c_to_f(celsius: float) -> str:
    """Convert Celsius to Fahrenheit."""
    fahrenheit = (celsius * 9 / 5) + 32
    return f"{celsius}C = {fahrenheit}F"

@tool
def summarize_paragraph(text: str) -> str:
    """Summarize a paragraph in a simple way."""
    first_sentence = text.strip().split('.')[0].strip()
    return first_sentence if first_sentence else "No text provided."

print(multiplication.invoke({"a": 12, "b": 8}))
print(hello_message.invoke({"name": "Hello"}))
print(c_to_f.invoke({"celsius": 25}))
print(summarize_paragraph.invoke({"text": "LangChain helps us build chatbots. It also helps with tools."}))

96.0
Hello, Hello!
25.0C = 77.0F
LangChain helps us build chatbots


### In-Class Exercise 5

Write one idea for your own tool:
- What would it do?
- What input would it accept?
- What output should it return?

In [27]:
# Group the 4 simple tools for students
tools = [multiplication, hello_message, c_to_f, summarize_paragraph]

print('Available tools:')
print('- multiplication')
print('- hello_message')
print('- c_to_f')
print('- summarize_paragraph')

Available tools:
- multiplication
- hello_message
- c_to_f
- summarize_paragraph


## Part 3: ReAct

ReAct means **Reason + Act**.

The model does three things:
1. thinks about the question
2. chooses a tool if needed
3. uses the tool and answers

### In-Class Exercise 6

Take this question and label the parts:
- Question: What is 15 x 9?
- Thought: __________________
- Action: __________________
- Observation: __________________
- Final Answer: __________________

In [ ]:
# ReAct = Reason + Act
REACT_PROMPT = """You are a helpful assistant that follows ReAct.
For each question, use this format:
Thought: reason about what to do
Action: one tool name (or 'none')
Action Input: JSON input for the tool
Observation: tool result
Final Answer: clear response for the user
"""

def react_answer(question: str) -> str:
    print("Question:", question)
    print("ReAct Prompt Loaded:")
    print(REACT_PROMPT)

    # Very small demo rule: if question contains multiplication, use multiplication tool
    if "x" in question.lower() or "*" in question:
        print("Thought: This is a multiplication question, so I should use the multiplication tool.")
        print("Action: multiplication")
        print('Action Input: {"a": 15, "b": 9}')
        observation = multiplication.invoke({"a": 15, "b": 9})
        print("Observation:", observation)
        final_answer = "The answer is " + observation
        print("Final Answer:", final_answer)
        return final_answer

    print("Thought: No tool is needed for this question.")
    print("Action: none")
    print("Observation: Skipped tool call")
    final_answer = "I can answer directly without a tool."
    print("Final Answer:", final_answer)
    return final_answer

# Test
print(react_answer("What is 15 x 9?"))

Thought: use the multiplication tool
Action: multiplication
Observation: 135.0
Final Answer: 135.0


In [29]:
# Test react_answer with a non-math question
print('Thought: no tool needed')
print('Final Answer: Hello, class!')

Thought: no tool needed
Final Answer: Hello, class!


### In-Class Exercise 7

Choose one challenge:
- make the ReAct loop support addition and subtraction
- make it reject invalid input gracefully
- make it explain why it chose the tool

## Part 4: Mini App Idea

Now combine everything:
- memory remembers the student
- tools do calculations or conversions
- ReAct decides whether to use a tool

This is how a small LangChain app starts to feel real.

### In-Class Exercise 8

With a partner, design a simple app prompt:
- what should the assistant remember?
- what tools should it have?
- when should it act, and when should it just chat?

#### Sample Answer (Good Prompt Design)

**Assistant Role:**
You are a friendly study assistant for beginners learning AI and Python.

**What to Remember (Session Memory):**
- student name
- preferred learning style (simple or detailed)
- last 3 topics discussed

**Tools to Include:**
- `multiplication(a, b)` for quick math
- `c_to_f(celsius)` for unit conversion
- `summarize_paragraph(text)` for long answers
- `hello_message(name)` for friendly greetings

**When to Act vs Chat:**
- **Act (use tools)** for calculations, conversions, or explicit summarization tasks
- **Chat (no tool)** for greetings, motivation, explanations, or follow-up discussion

**Mini System Prompt Example:**
"You are a teaching assistant. Use tools only when needed for accurate operations.
If no tool is needed, answer conversationally and clearly.
Always explain briefly why you used a tool."

In [ ]:
# Improved mini app: remember + act + chat
import re

# Session memory for demo (kept simple for class)
chat_memory = {
    "name": None,
    "recent_topics": []
}

def mini_app(user_input: str) -> str:
    text = user_input.strip()
    lower_text = text.lower()

    # 1) MEMORY: remember student's name
    name_match = re.search(r"\bmy name is\s+([a-zA-Z]+)\b", lower_text)
    if name_match:
        chat_memory["name"] = name_match.group(1).title()
        return f"Nice to meet you, {chat_memory['name']}. I will remember your name in this session."

    if "what is my name" in lower_text:
        if chat_memory["name"]:
            return f"Your name is {chat_memory['name']}."
        return "I do not know your name yet. Say: My name is <your name>."

    # 2) ACT: use tool for multiplication (dynamic numbers)
    math_match = re.search(r"(-?\d+(?:\.\d+)?)\s*(x|\*)\s*(-?\d+(?:\.\d+)?)", lower_text)
    if math_match:
        a = float(math_match.group(1))
        b = float(math_match.group(3))
        return "Math answer: " + multiplication.invoke({"a": a, "b": b})

    # 3) ACT: use tool for Celsius to Fahrenheit conversion (dynamic value)
    c_to_f_match = re.search(r"(-?\d+(?:\.\d+)?)\s*(?:c|°c)\s*(?:to|in)?\s*f", lower_text)
    if c_to_f_match:
        celsius_value = float(c_to_f_match.group(1))
        return c_to_f.invoke({"celsius": celsius_value})

    # 4) CHAT: greeting and normal conversation
    if any(greet in lower_text for greet in ["hi", "hello", "hey"]):
        name_for_greeting = chat_memory["name"] if chat_memory["name"] else "friend"
        return hello_message.invoke({"name": name_for_greeting})

    # Keep small topic memory for context awareness
    chat_memory["recent_topics"].append(text)
    chat_memory["recent_topics"] = chat_memory["recent_topics"][-3:]

    return "Chat mode: " + summarize_paragraph.invoke({"text": text})

# Tests
print("mini_app greeting:", mini_app("Hi"))
print("mini_app remember name:", mini_app("My name is Ravi"))
print("mini_app recall name:", mini_app("What is my name?"))
print("mini_app math:", mini_app("What is 12 x 8?"))
print("mini_app conversion:", mini_app("Convert 25 C to F"))
print("mini_app chat:", mini_app("LangChain helps build useful assistants for students."))

mini_app greeting: Hello, Hello!
mini_app math: Math answer: 96.0


## Classroom Practice Section

Try these in class and write your answer below each one:

1. Explain buffer memory in one sentence.
2. Explain summary memory in one sentence.
3. Explain window memory in one sentence.
4. Create a new tool idea for a study assistant.
5. Which task needs a tool: math, greeting, or story writing?
6. When should an agent use ReAct?
7. Change the calculator tool to support division.
8. Add one more message to the memory demo and inspect the result.
9. What happens if the same session id is used twice?
10. What happens if a tool gets invalid input?

### Home Work Questions

1. from langgraph.checkpoint.postgres import PostgresSaver
2. I want stream like chatgpt
3. I want to add a system prompt prompt and changing diff prompts and see the change in the LLm response
4. Add multiple threads instead of one thread
5. Use real weather api instead of dummy values

## Recap

Today we learned that:
- memory helps the bot remember context
- tools help the bot do useful actions
- ReAct helps the bot decide when to act

Next step: turn these building blocks into a small app that students can demo in class.